# Laboratorio de regresión - 5

|                |   |
:----------------|---|
| **Nombre**     |Santiago Escutia Ríos   |
| **Fecha**      | 16/2/2026  |
| **Expediente** |  757839 |

## Validación

Hemos estado usando `train_test_split` en nuestros modelos anteriores.

¿Por qué?

**Porque de esa manera se hace una prueba con los datos para ver si nuestras regresiones son buenas para después usarlas con la totalidad de los datos.**

Si la muestra es un subset de la población y queremos generalizar sobre la población, ¿no sería mejor utilizar todos los datos al entrenar un modelo?

**No, porque lo que se ocupa es que divides los datos y con eso lo entrenas y ya depues le metes los otros y ya con eso ves si se generaliza.**

El propósito de volver a muestrear dentro de nuestro dataset es tener una idea de qué tan buena podría ser la generalización de nuestro modelo. Imagina un dataset ya separado en dos mitades. Utilizas la primera mitad para entrenar el modelo y pruebas en la segunda mitad; la segunda mitad eran datos invisibles para el modelo al momento de entrenar. Esto nos lleva a tres escenario típicos:

1. Si el modelo hace buenas predicciones en la segunda mitad, significa que la primera mitad era "suficiente" para generalizar.
2. Si el modelo no hace buenas predicciones en la segunda mitad, pero sí en la primera mitad, podría ser que había información importante en la segunda mitad que debió haber sido tomada en cuenta al entrenar, o un problema de overfitting.
3. Si el modelo no hace buenas predicciones en la segunda mitad, y tampoco en la primera mitad, se tendrían que revisar los factores y/o el modelo seleccionado.

El caso ideal sería el 1, pero por estadística los errores y varianzas tienen como entrada el número de muestas, por lo que tenemos menos seguridad de nuestros resutados al usar menos muestras. Si vemos que el modelo generaliza bien podemos unir de nuevo el dataset y entrenar sobre el dataset completo.

En el caso 2 está el problema de que no podemos saber qué información es necesaria para el entrenamiento apropiado del modelo; esto nos lleva a pensar que debemos usar el dataset completo para entrenar, pero esto nos lleva al mismo problema de no saber si el modelo puede generalizar.

El problema sólo incrementa si se tienen hiperparámetros en el modelo (e.g. $\lambda$ en regularización).

## Leave-One-Out Cross Validation

Este método de validación es una colección de $n$ `train-test-split`. Teniendo un dataset de $n$ muestras, la lógica es:
1. Saca una muestra del dataset.
2. Entrena tu modelo con las $n-1$ muestras.
3. Evalúa tu modelo en la muestra que quedó fuera con el métrico que más se ajuste a la aplicación.
4. Regresa la muestra al dataset.
5. Repite 1-4 con muestras diferentes hasta haber hecho el procedimiento $n$ veces para $n$ muestras.
6. Calcula la media y desviación estándar de los métricos guardados.

Con los resultados del proceso de validación podemos saber qué tan bueno podría ser el modelo seleccionado con los datos (con/sin transformaciones).

### Ejercicio 1

Utiliza el dataset `Motor Trend Car Road Tests`. Elimina la columna `model` y entrena 32 modelos diferentes utilizando Leave-One-Out Cross Validation con target `mpg`. Utiliza MSE como métrico.

In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


df= pd.read_excel("Motor Trend Car Road Tests.xlsx")
df_clean = df.drop(columns=['model'])

X = df_clean.drop(columns=['mpg'])
y = df_clean['mpg']

In [16]:
errores_mse = []

# Determinamos cuántas filas hay (n = 32)
n = len(df_clean)

#Ciclo manual
for i in range(n):
    # PASO A: SEPARACIÓN 
    # Usamos el índice 'i' para sacar una sola fila para prueba
    X_test = X.iloc[[i]]
    y_test = y.iloc[[i]]
    
    # Usamos todos los demás índices para entrenamiento
    X_train = X.drop(i)
    y_train = y.drop(i)
    
    # PASO B: ENTRENAMIENTO 
    modelo = LinearRegression()
    modelo.fit(X_train, y_train)
    
    # PASO C: PREDICCIÓN 
    # El modelo intenta adivinar el mpg del auto que dejamos fuera
    prediccion = modelo.predict(X_test)
    
    # PASO D: CALCULAR EL ERROR 
    # Calculamos (Real - Predicho)^2
    error_individual = (y_test.values[0] - prediccion[0])**2
    errores_mse.append(error_individual)

In [15]:
#Resultados finales (Media y Desviación Estándar)
promedio_mse = np.mean(errores_mse)
desviacion_mse = np.std(errores_mse)

print(f"MSE promedio: {promedio_mse:.4f}")
print(f"Desviación estándar: {desviacion_mse:.4f}")

MSE promedio: 12.1816
Desviación estándar: 17.0674


Interpreta.

El modelo entiende la tendencia general, pero es poco estable. La desviación alta indica que 32 datos no son suficientes para que el modelo aprenda las excepciones, por lo que funciona bien con autos promedio pero falla mucho con los casos más raros o extremos.

## K-Folds Cross-Validation

El dataset `Motor Trend Car Road Tests` sólo tiene 32 muestras, y utilizar un modelo sencillo de regresión múltiple hace que usar LOOCV sea muy rápido. El dataset `California Housing` tiene $20640$ muestras para $9$ columnas, entonces realizar un ajuste sobre una transformación o sobre el modelo y luego calcular el impacto esperado podría tomar más tiempo.

La solución propuesta es dividir el dataset en *k* folds (partes iguales), ajustar en *k-1* folds y probar en el restante.

### Ejercicio 2
Utiliza el dataset `California Housing` y haz K-folds Cross Validation con 10 folds. Utiliza el MSE como métrico.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print("Dataset Shape:", housing.data.shape, housing.target.shape)
print("Dataset Features:", housing.feature_names)
print("Dataset Target:", housing.target_names)
X = housing.data
y = housing.target

Interpreta.

## Referencia

James, G., Witten, D., Hastie, T., Tibshirani, R.,, Taylor, J. (2023). An Introduction to Statistical Learning with Applications in Python. Cham: Springer. ISBN: 978-3-031-38746-3